# ROUTING

## ENVIRONMENT

In [ ]:
! pip install langchain_community tiktoken langchain-openai langchainhub chromadb langchain

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ['LANGCHAIN_TRACING_V2'] = 'true'
os.environ['LANGCHAIN_ENDPOINT'] = 'https://api.smith.langchain.com'
if os.getenv("LANGCHAIN_API_KEY"):
    os.environ['LANGCHAIN_API_KEY'] = os.getenv("LANGCHAIN_API_KEY")
if os.getenv("OPENAI_API_KEY"):
    os.environ['OPENAI_API_KEY'] = os.getenv("OPENAI_API_KEY")
os.environ['OPENAI_API_BASE'] = 'https://openrouter.ai/api/v1'
os.environ['OPENAI_BASE_URL'] = 'https://openrouter.ai/api/v1'

#### LOGICAL ROUTING

LLM reads the question and decides with reasoning.

In [ ]:
from typing import Literal

from langchain_core.prompts import ChatPromptTemplate
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI

from langchain_core.runnables import RunnableLambda

In [ ]:
class RouteQuery(BaseModel):
    """Route a user query to the most relevant datasource."""

    datasource: Literal["python_docs", "js_docs", "golang_docs"] = Field(
        ...,
        description="Given a user question choose which datasource would be most relevant for answering their question",
    )

Here we are defining the output format for llm.

In [ ]:
llm = ChatOpenAI(model="nex-agi/nex-n2-pro:free",temperature=0)
structured_llm = llm.with_structured_output(RouteQuery)

Normal LLM returns plain text this makes it return a structured object.

In [ ]:

system = """You are an expert at routing a user question to the appropriate data source.
Based on the programming language the question is referring to, route it to the relevant data source."""

prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)

router = prompt | structured_llm

Here we are providing the prompt and router return which type of datasource to use.

In [ ]:
question = """Why doesn't the following code work:
prompt = ChatPromptTemplate.from_messages(["human", "speak in {language}"])
prompt.invoke("french")
"""

result = router.invoke({"question": question})

In this block we ask question and use router to find the datasource.

In [ ]:
def choose_route(result):
    if "python_docs" in result.datasource.lower():
        return "chain for python_docs"
    elif "js_docs" in result.datasource.lower():
        return "chain for js_docs"
    else:
        return "golang_docs"
    
full_chain = router | RunnableLambda(choose_route)
full_chain.invoke({"question": question})

Once we find the data source we print it using the function choose_route.

### How Logical Routing Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       LOGICAL ROUTING                                │
└─────────────────────────────────────────────────────────────────────────┘

  ┌────────────────────┐
  │  User Question       │
  └──────────┬─────────┘
             │
             ▼
  ┌────────────────────────────────┐
  │  LLM reads question             │
  │  (Structured Output: RouteQuery) │
  └────────────────┬───────────────┘
                   │
       ┌──────────┼──────────┐
       │           │           │
       ▼           ▼           ▼
  ┌─────────┐ ┌────────┐ ┌───────────┐
  │ python   │ │ js     │ │ golang     │
  │ _docs    │ │ _docs  │ │ _docs      │
  └────┬────┘ └───┬────┘ └─────┬─────┘
       │          │           │
       └──────────┴──────────┘
                   │
                   ▼
  ┌────────────────────────────────┐
  │  Run corresponding RAG Chain    │
  └────────────────┬───────────────┘
                   │
                   ▼
  ┌───────────────────────┐
  │  Final Answer          │
  └───────────────────────┘
```

**Key Insight:**
- LLM uses **reasoning** to decide the right datasource based on question content
- Returns a **structured output** (RouteQuery) with the selected datasource
- Different datasources run different specialized RAG chains


#### SEMENTIC ROUTING

Embeddings decide based on similarity. No LLM needed for routing.

In [ ]:
from langchain_community.utils.math import cosine_similarity
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableLambda, RunnablePassthrough
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

In [ ]:
physics_template = """You are a very smart physics professor. \
You are great at answering questions about physics in a concise and easy to understand manner. \
When you don't know the answer to a question you admit that you don't know.

Here is a question:
{query}"""

math_template = """You are a very good mathematician. You are great at answering math questions. \
You are so good because you are able to break down hard problems into their component parts, \
answer the component parts, and then put them together to answer the broader question.

Here is a question:
{query}"""

Here we are giving two prompt and it will decide according to question.

In [ ]:
embeddings = OpenAIEmbeddings()
prompt_templates = [physics_template, math_template]
prompt_embeddings = embeddings.embed_documents(prompt_templates)

Here we form embedding for both prompt.

In [ ]:
def prompt_router(input):
    query_embedding = embeddings.embed_query(input["query"])
    similarity = cosine_similarity([query_embedding], prompt_embeddings)[0]
    most_similar = prompt_templates[similarity.argmax()]
    print("Using MATH" if most_similar == math_template else "Using PHYSICS")
    return PromptTemplate.from_template(most_similar)

In this block of code we find the cosine similarity between question and both the prompt the most similar prompt will use.

In [ ]:
chain = (
    {"query": RunnablePassthrough()}
    | RunnableLambda(prompt_router)
    | ChatOpenAI()
    | StrOutputParser()
)

print(chain.invoke("What's a black hole"))

This is the final stepp were we connect all and hense find both datasource and answer to the query.

### How Semantic Routing Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       SEMANTIC ROUTING                                │
└─────────────────────────────────────────────────────────────────────────┘

  ┌────────────────────┐
  │  User Question       │
  └──────────┬─────────┘
             │
             ▼
  ┌────────────────────────────────┐
  │  Embed Question                │
  │ (OpenAIEmbeddings)             │
  └────────────────┬───────────────┘
                   │
                   ▼
  ┌────────────────────────────────────────┐
  │  Cosine Similarity with prompt embeddings │
  │ (Physics prompt vs Math prompt)           │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌────────────────────────────────────────┐
  │  Select most similar prompt               │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌────────────────────────────────────────┐
  │  LLM answers using selected prompt        │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌───────────────────────┐
  │  Final Answer          │
  └───────────────────────┘
```

**Key Insight:**
- **No LLM needed for routing** — embeddings decide based on similarity
- Question is embedded and compared against pre-embedded prompt templates
- The most similar prompt template is automatically selected


# QUERY CONSTRUCTION

 It convert a natural language question into a structured database query with filters

In [ ]:
from langchain_community.document_loaders import YoutubeLoader

docs = YoutubeLoader.from_youtube_url(
    "https://www.youtube.com/watch?v=pbAd8O1Lvm4", add_video_info=True
).load()

docs[0].metadata

Here we give te youtube video database.

In [ ]:
import datetime
from typing import Literal, Optional, Tuple
from pydantic import BaseModel, Field

class TutorialSearch(BaseModel):
    """Search over a database of tutorial videos about a software library."""

    content_search: str = Field(
        ...,
        description="Similarity search query applied to video transcripts.",
    )
    title_search: str = Field(
        ...,
        description=(
            "Alternate version of the content search query to apply to video titles. "
            "Should be succinct and only include key words that could be in a video "
            "title."
        ),
    )
    min_view_count: Optional[int] = Field(
        None,
        description="Minimum view count filter, inclusive. Only use if explicitly specified.",
    )
    max_view_count: Optional[int] = Field(
        None,
        description="Maximum view count filter, exclusive. Only use if explicitly specified.",
    )
    earliest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Earliest publish date filter, inclusive. Only use if explicitly specified.",
    )
    latest_publish_date: Optional[datetime.date] = Field(
        None,
        description="Latest publish date filter, exclusive. Only use if explicitly specified.",
    )
    min_length_sec: Optional[int] = Field(
        None,
        description="Minimum video length in seconds, inclusive. Only use if explicitly specified.",
    )
    max_length_sec: Optional[int] = Field(
        None,
        description="Maximum video length in seconds, exclusive. Only use if explicitly specified.",
    )
    
    def pretty_print(self) -> None:
        for field_name, field_info in self.__class__.model_fields.items():
            value = getattr(self, field_name)
            if value is not None and value != field_info.default:
                print(f"{field_name}: {value}")

Here we define the class. This is like a form that llm must fill.Some are optional to fill.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

system = """You are an expert at converting user questions into database queries. \
You have access to a database of tutorial videos about a software library for building LLM-powered applications. \
Given a question, return a database query optimized to retrieve the most relevant results.

If there are acronyms or words you are not familiar with, do not try to rephrase them."""
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", system),
        ("human", "{question}"),
    ]
)
llm = ChatOpenAI(model="nex-agi/nex-n2-pro:free", temperature=0)
structured_llm = llm.with_structured_output(TutorialSearch)
query_analyzer = prompt | structured_llm


Here we define the system and then query is asked in simple English  .LLM read that query and fill the tutorial search schema, and then it finally prepared a structured query with filters.

In [ ]:
query_analyzer.invoke(
    {
        "question": "how to use multi-modal models in an agent, only videos under 5 minutes"
    }
).pretty_print()

In [ ]:
query_analyzer.invoke(
    {"question": "videos that are focused on the topic of chat langchain that are published before 2024"}
).pretty_print()

In [ ]:
query_analyzer.invoke(
    {"question": "videos on chat langchain published in 2023"}
).pretty_print()

### How Query Construction Works

```
┌─────────────────────────────────────────────────────────────────────────┐
│                       QUERY CONSTRUCTION                              │
└─────────────────────────────────────────────────────────────────────────┘

  ┌────────────────────────────────────────┐
  │  User Question (natural language)          │
  │  "videos on chat langchain published 2023" │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌────────────────────────────────────────┐
  │  LLM reads question                        │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌────────────────────────────────────────┐
  │  Fills Pydantic Schema (TutorialSearch)    │
  │  content_search, title_search,             │
  │  min_view_count, publish_date, etc.        │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌────────────────────────────────────────┐
  │  Structured Query with filters             │
  └───────────────────┬────────────────────┘
                      │
                      ▼
  ┌────────────────────────────────────────┐
  │  Database Search ─► Precise Results         │
  └────────────────────────────────────────┘
```

**Key Insight:**
- LLM converts **natural language to structured query** — no manual filter writing needed
- Uses **Pydantic schema** to define the query structure
- Enables precise database searches with filters (dates, view counts, etc.)
